# PANDUAN CLEANING DATA UNTUK MENGHITUNG PKB PER TAHUN

## OLEH: DHINDA TSAMARA SHALSABILLA

### Import Library yang Digunakan

#### Bila belum menginstall library, maka install terlebih dahulu di cmd ataupun terminal VS Code dengan mengetik "pip install pandas numpy"

In [ ]:
import pandas as pd
import numpy as np

### Load Data

#### Ganti nama/path file yang akan dibersihkan. Jika file CSV berada di folder yang sama dengan code ini, maka cukup masukkan nama file saja. Namun, jika file CSV berada di folder yang berbeda, maka copy path dari file tersebut. Pastikan file dalam bentuk CSV.

In [ ]:
df = pd.read_csv("nama_file_atau_path.csv")
print(f"Data berhasil dimuat dengan {df.shape[0]} baris dan {df.shape[1]} kolom.")
df

#### Melihat tipe data dari setiap kolom

In [ ]:
df.info()

### Memilih Kolom yang Dibutuhkan

#### Masukkan nama kolom yang tidak dibutuhkan ke dalam variabel "drop", maka nantinya kolom tersebut akan dihapus dari data

In [ ]:
drop = ["ntc_tb_bbn1_p", "ntc_tb_bbn1_d", "ntc_tb_bbn2_p", "ntc_tb_bbn2_d", "ntc_dasar_pkb"]  
df.drop(columns = drop, axis = 1, inplace = True, errors = "ignore")
df.head()

#### Membuat kolom baru dan menambahkan di bagian paling akhir

In [ ]:
df["total_pkb"] = df["ntc_tb_pkb_p"] + df["ntc_tg_pkb_p"]

In [ ]:
df["is_late"] = df.apply(lambda row: 1 if (row["ntc_tg_tahun"] > 0) or 
                                       (row["ntc_tg_bulan"] > 0) or 
                                       (row["ntc_tg_hari"] > 0) else 0, axis=1)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Convert Datetime Type

#### Mengubah tipe data untuk kolom tanggal menjadi datetime karena nantinya akan dilakukan operasi untuk menghitung tanggal

In [ ]:
date_cols = ["bnk_tanggal", "ctk_notice_tanggal", "knd_tgl_notice_old", "knd_tgl_notice_new", "knd_tgl_stnk_start", "knd_tgl_stnk_end"]
for col_name in date_cols:
    df[col_name] = pd.to_datetime(df[col_name], errors='coerce')

### Drop Duplicates

#### Menghapus data duplikat yang ada di dalam data

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()
print(f"Setelah menghapus duplikasi, data memiliki {df.shape[0]} baris.")

### Missing Values

#### Mengecek nilai-nilai kosong yang ada di dalam data

In [ ]:
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

#### Berdasarkan hasil pengecekan di atas, maka kita bisa untuk mengisi nilai-nilai kosong tersebut. Handling missing values ini disesuaikan dengan setiap variabel yang memiliki nilai kosong berdasarkan hasil pengecekan. Cara untuk handling missing values berbeda-beda tiap kolom, namun alurnya tetap sama.

##### Handling Missing Values 'knd_tgl_notice_old'

##### Mengecek ada berapa missing values untuk kolom knd_tgl_notice_old

In [ ]:
df["knd_tgl_notice_old"].isnull().sum()  
df[df["knd_tgl_notice_old"].isnull()]    

##### Mengisi missing values dari kolom knd_tgl_notice_old dengan mengurangkan 1 tahun dari kolom knd_tgl_notice_new

In [ ]:
df["knd_tgl_notice_old"] = df["knd_tgl_notice_old"].fillna(df["knd_tgl_notice_new"] - pd.DateOffset(years=1))

##### Mengecek apakah code sebelumnya berjalan dengan benar dengan mencari id dari data kosong yang ada di knd_tgl_notice_old. Nilai dari row_id dapat disesuaikan dengan melihat id ketika mengecek baris mana yang kosong di knd_tgl_notice_old

In [ ]:
pd.set_option("display.max_columns", None)
df.loc[df["row_id"] == 7552823]

##### Handling Missing Values 'knd_tgl_notice_new'

##### Mengecek ada berapa missing values untuk kolom knd_tgl_notice_new

In [ ]:
df["knd_tgl_notice_new"].isnull().sum()  
df[df["knd_tgl_notice_new"].isnull()]    

##### Mengisi missing values dari kolom knd_tgl_notice_new dengan menambahkan 1 tahun dari kolom knd_tgl_notice_old

In [ ]:
df["knd_tgl_notice_new"] = df["knd_tgl_notice_new"].fillna(df["knd_tgl_notice_old"] + pd.DateOffset(years=1))

##### Mengecek apakah code sebelumnya berjalan dengan benar dengan mencari id dari data kosong yang ada di knd_tgl_notice_new. Nilai dari row_id dapat disesuaikan dengan melihat id ketika mengecek baris mana yang kosong di knd_tgl_notice_new

In [ ]:
pd.set_option("display.max_columns", None)
df.loc[df["row_id"] == 7552823]

##### Handling Missing Values 'knd_tgl_stnk_start'

In [ ]:
df["knd_tgl_stnk_start"].isnull().sum()  
df[df["knd_tgl_stnk_start"].isnull()]    

##### Menangani missing values dan juga memperbaiki data dari knd_tgl_stnk_start dengan mengurangi 5 tahun dari kolom knd_tgl_stnk_end

In [ ]:
df.loc[df["knd_tgl_stnk_start"].isin(["1900-01-01"]) | df["knd_tgl_stnk_start"].isna(), "knd_tgl_stnk_start"] = df["knd_tgl_stnk_end"] - pd.DateOffset(years=5)

In [ ]:
df.loc[df["row_id"] == 7630664]

### Handling Missing Values 'knd_tgl_stnk_end'

In [ ]:
df["knd_tgl_stnk_end"].isnull().sum()  
df[df["knd_tgl_stnk_end"].isnull()]    

##### Menangani missing values dan juga memperbaiki data dari knd_tgl_stnk_end dengan menambahkan 5 tahun dari kolom knd_tgl_stnk_start

In [ ]:
df.loc[df["knd_tgl_stnk_end"].gt("2030-12-01") | df["knd_tgl_stnk_end"].isna(), "knd_tgl_stnk_end"] = df["knd_tgl_stnk_start"] + pd.DateOffset(years=5)

In [ ]:
df.loc[df["row_id"] == 7632766]

#### Check Missing Values Again

In [ ]:
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

### Convert Data Type and Add New Columns

#### Mengubah tipe data tiap kolom agar sesuai sehingga bisa diolah lebih lanjut

##### Membuat kolom baru untuk mengecek apakah telat membayar dan jumlah hari keterlambatan

In [ ]:
df["keterlambatan"] = np.where(
    (df["bnk_tanggal"] >= df["knd_tgl_notice_old"]),
    (df["bnk_tanggal"] - df["knd_tgl_notice_old"]).dt.days,
    0
)

In [ ]:
df["ketepatan"] = (df["bnk_tanggal"] - df["knd_tgl_notice_old"]).dt.days

##### Masukkan kolom-kolom yang seharusnya bertipe int (angka dan tidak mengandung koma)

In [ ]:
int_cols = ["ntc_tb_pkb_p", "ntc_tb_pkb_d", "ntc_tg_tahun", "ntc_tg_bulan", 
            "ntc_tg_hari", "ntc_tg_pkb_p", "ntc_tg_pkb_d", "total_pkb", 
            "keterlambatan", "ketepatan"]
for col_name in int_cols:
    df[col_name] = df[col_name].astype("Int64")

In [ ]:
df.info()

In [ ]:
df.head()

### Save the Data

#### Ganti nama file/path yang akan digunakan untuk menyimpan data di bagian "df.to_csv()"
#### Apabila ingin menyimpan file CSV di dalam folder yang sama dengan code ini berada, maka cukup tuliskan nama file saja, namun jika ingin menyimpan di folder yang berbeda, maka copy path tujuan yang disertai dengan nama file yang berekstensi .csv

In [ ]:
df.to_csv("nama_file_atau_path.csv", index=False)
print("Data berhasil disimpan")